In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructors_df = (
    spark.table(f"{catalog_name}.{silver_schema}.constructors")
         .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
dim_constructors_df = (
    constructors_df.alias("ct")
        .join(
            ref_nationality_region_df.alias("nrg")
            , F.col("ct.nationality") == F.col("nrg.nationality")
            , "left"
        )
        .selectExpr(
            "ct.constructor_id"
            , "ct.constructor_name"
            , "ct.nationality"
            , "nrg.region as nationality_region"
        )
)
display(dim_constructors_df)

In [0]:
write_to_gold(
    input_df=dim_constructors_df,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
display(spark.table(target_table))